# **Milestone 4**

## **Introduction**

This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.


Suggested Readings:
* Hugging Face Multiple Choice Task

* LoRA Documentation

* PyTorch Softmax

Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

Perform the following tasks on the dataset provided as part of the Kaggle competition.

## **Initialization**

In [1]:
!pip install --upgrade torchao peft

In [2]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

train = pd.read_csv("/content/train.csv")

## **Question 01**

Label Encoding

Convert the answer column in train.csv into numeric labels using the following mapping:  
A = 0  
B = 1  
C = 2  
D = 3  
E = 4  

What is the encoded numeric label for the row at index 150?

In [3]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
print(label_map[train.iloc[150]["answer"]])

2


## **Question 02**

Prompt-Option Formatting  
For row index 0, create the Option B input using exactly this format:  
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?


In [4]:
prompt_0 = str(train.iloc[0]["prompt"])
opt_B_0 = str(train.iloc[0]["B"])
formatted_str = prompt_0 + " [SEP] " + opt_B_0
print(len(formatted_str))

407


## **Question 03**

Single-Row MCQ Tokenization  
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:  
padding = "max_length"  
truncation = True  
max_length = 128  
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:  
[1, 5, 128]

What is the value of the second dimension?

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
options_0 = [str(train.iloc[0][l]) for l in ["A", "B", "C", "D", "E"]]
prompts_0 = [prompt_0] * 5

tokens_0 = tokenizer(prompts_0, options_0, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids = tokens_0["input_ids"].unsqueeze(0)
attention_mask = tokens_0["attention_mask"].unsqueeze(0)
print(input_ids.shape[1])

5


## **Question 04**

Batch MCQ Tokenization  
Tokenize the first 16 rows of train.csv as multiple-choice examples.  
Each row has 5 choices.  
Each choice is tokenized to length 128.

The final input_ids tensor has shape:  
[16, 5, 128]

How many total token positions are in this tensor?

In [6]:
print(16 * 5 * 128)

10240


## **Question 05**

Multiple-Choice Logits  
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:  
[1, 5]

How many logits are produced for one question?

In [7]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

with torch.no_grad():
    outputs_5 = model(input_ids=input_ids, attention_mask=attention_mask)

print(outputs_5.logits.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5


## **Question 06**

Supervised Loss Tensor  
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [8]:
label_0 = torch.tensor([label_map[train.iloc[0]["answer"]]])

with torch.no_grad():
    outputs_6 = model(input_ids=input_ids, attention_mask=attention_mask, labels=label_0)

print(outputs_6.loss.dim())

0


## **Question 07**

LoRA for Efficient Fine-Tuning
LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

LoRA Trainable Parameters  
Apply LoRA to the bert-base-uncased multiple-choice model using:  
r = 8  
lora_alpha = 16  
target_modules = ["query", "value"]  
lora_dropout = 0.1  
bias = "none"  
task_type = TaskType.SEQ_CLS

Count trainable parameters using:  
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [9]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, peft_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params)

295681


## **Question 08**

Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Hugging Face Dataset Preparation  
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:  
input_ids with shape [5, 128]  
attention_mask with shape [5, 128]  
labels as the encoded answer label

For the first dataset item, input_ids has shape:  
[5, 128]

How many tokenized choices are stored in input_ids?

In [10]:
def preprocess_8(examples):
    prompts = [[context] * 5 for context in examples["prompt"]]
    options = [[str(examples[opt][i]) for opt in ["A", "B", "C", "D", "E"]] for i in range(len(examples["prompt"]))]

    prompts = sum(prompts, [])
    options = sum(options, [])

    tokenized = tokenizer(prompts, options, padding="max_length", truncation=True, max_length=128)

    return {
        "input_ids": [tokenized["input_ids"][i:i+5] for i in range(0, len(tokenized["input_ids"]), 5)],
        "attention_mask": [tokenized["attention_mask"][i:i+5] for i in range(0, len(tokenized["attention_mask"]), 5)],
        "labels": [label_map[ans] for ans in examples["answer"]]
    }

train_100 = train.iloc[:100].copy()
train_100.fillna("", inplace=True)
dataset_100 = Dataset.from_pandas(train_100)
tokenized_100 = dataset_100.map(preprocess_8, batched=True)

print(len(tokenized_100[0]["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

5


## **Question 09**

Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Tiny LoRA Fine-Tuning  
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:  
max_length = 64  
per_device_train_batch_size = 4  
gradient_accumulation_steps = 1  
max_steps = 4

What is the final global_step reported by the Trainer?

In [11]:
def preprocess_9(examples):
    prompts = [[context] * 5 for context in examples["prompt"]]
    options = [[str(examples[opt][i]) for opt in ["A", "B", "C", "D", "E"]] for i in range(len(examples["prompt"]))]

    prompts = sum(prompts, [])
    options = sum(options, [])

    tokenized = tokenizer(prompts, options, padding="max_length", truncation=True, max_length=64)

    return {
        "input_ids": [tokenized["input_ids"][i:i+5] for i in range(0, len(tokenized["input_ids"]), 5)],
        "attention_mask": [tokenized["attention_mask"][i:i+5] for i in range(0, len(tokenized["attention_mask"]), 5)],
        "labels": [label_map[ans] for ans in examples["answer"]]
    }

train_32 = train.iloc[:32].copy()
train_32.fillna("", inplace=True)
dataset_32 = Dataset.from_pandas(train_32)
tokenized_32 = dataset_32.map(preprocess_9, batched=True, remove_columns=dataset_32.column_names)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_32,
)

train_result = trainer.train()
print(train_result.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.658613
2,1.563974
3,1.671073
4,1.703346


4


## **Question 10**

Probability Assigned to Option E After Fine-Tuning  
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [12]:
row_0 = tokenized_32[0]
input_ids_0 = torch.tensor([row_0["input_ids"]]).to(model.device)
attention_mask_0 = torch.tensor([row_0["attention_mask"]]).to(model.device)

model.eval()
with torch.no_grad():
    outputs_10 = model(input_ids=input_ids_0, attention_mask=attention_mask_0)

probs = F.softmax(outputs_10.logits, dim=-1)
prob_E = probs[0][4].item()
print(round(prob_E, 4))

0.2031
